# FIFA World Cup 2026 - Modelo de Prediccion de Marcadores
## Modelo: Dixon-Coles con Poisson bivariante

### Variables del modelo:
| Variable | Descripcion | Fuente |
|---|---|---|
| **Ataque (λ_h)** | Fuerza ofensiva historica ponderada por recencia | Resultados internacionales |
| **Defensa (λ_a)** | Solidez defensiva historica ponderada por recencia | Resultados internacionales |
| **Ranking FIFA** | Posicion oficial FIFA (calibra parametros iniciales) | FIFA.com |
| **Rating Elo** | Sistema de rating similar al ajedrez | eloratings.net |
| **xG / xGA** | Expected Goals (mide calidad de ocasiones) | StatsBomb / FBref |
| **Forma reciente** | Ultimo 10-15 partidos (decaimiento exponencial) | Resultados |
| **H2H** | Historial cabeza a cabeza entre los dos equipos | Resultados |
| **Torneo neutral** | El Mundial se juega en terreno neutral | - |
| **Corrección ρ** | Ajuste Dixon-Coles para marcadores 0-0, 1-0, 0-1, 1-1 | Estadistico |

In [ ]:
import sys, warnings
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
from pathlib import Path

from src.data.preprocessing import (
    load_match_results, load_elo_ratings,
    prepare_training_data, compute_team_stats, compute_h2h_stats
)
from src.models.dixon_coles import DixonColesModel
from src.simulation.tournament import TournamentPredictor, load_tournament_data

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
print('Modulos cargados correctamente')

## 1. Carga y Exploracion de Datos

**Para mejores predicciones, descarga primero:**
- **Resultados historicos**: https://www.kaggle.com/datasets/martj42/international-football-results-from-1872-to-2017
- **Rankings FIFA**: https://www.kaggle.com/datasets/cashncarry/fifaworldranking
- **Elo Ratings**: https://www.eloratings.net/World.tsv

Guarda los archivos en `data/raw/`

In [ ]:
# Cargar datos (usa datos de ejemplo si no hay CSV descargado)
results_df = load_match_results()
elo_df = load_elo_ratings()

print(f'\nPartidos totales cargados: {len(results_df):,}')
print(f'Periodo: {results_df["date"].min().year} - {results_df["date"].max().year}')
results_df.head()

In [ ]:
# Distribucion de goles por partido
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Goles totales
total_goals = results_df['home_score'] + results_df['away_score']
axes[0].hist(total_goals, bins=range(0, 15), edgecolor='black', alpha=0.7, color='steelblue')
axes[0].set_title('Distribucion de Goles por Partido', fontsize=12)
axes[0].set_xlabel('Total de Goles')
axes[0].set_ylabel('Frecuencia')
axes[0].axvline(total_goals.mean(), color='red', linestyle='--', label=f'Media: {total_goals.mean():.2f}')
axes[0].legend()

# Goles local
axes[1].hist(results_df['home_score'], bins=range(0, 12), edgecolor='black', alpha=0.7, color='green')
axes[1].set_title('Goles del Equipo Local', fontsize=12)
axes[1].set_xlabel('Goles')
axes[1].axvline(results_df['home_score'].mean(), color='red', linestyle='--',
                label=f'Media: {results_df["home_score"].mean():.2f}')
axes[1].legend()

# Goles visitante
axes[2].hist(results_df['away_score'], bins=range(0, 12), edgecolor='black', alpha=0.7, color='orange')
axes[2].set_title('Goles del Equipo Visitante', fontsize=12)
axes[2].set_xlabel('Goles')
axes[2].axvline(results_df['away_score'].mean(), color='red', linestyle='--',
                label=f'Media: {results_df["away_score"].mean():.2f}')
axes[2].legend()

plt.tight_layout()
plt.savefig('../data/wc2026/distribucion_goles.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Preparacion del Dataset de Entrenamiento

In [ ]:
# Filtrar ultimos 8 años con pesos de recencia
train_df = prepare_training_data(results_df, years=8)

# Estadisticas por equipo
team_stats = compute_team_stats(train_df)
print(f'\nEquipos con datos suficientes: {len(team_stats)}')
print('\nTop 20 equipos por diferencia de gol promedio:')
team_stats.head(20)[['team', 'matches', 'goals_for_avg', 'goals_against_avg', 
                       'goal_diff_avg', 'win_rate', 'pts_last5']]

In [ ]:
# Visualizar fortaleza ofensiva vs defensiva
wc_teams = [
    'Argentina', 'France', 'England', 'Brazil', 'Spain', 'Portugal',
    'Germany', 'Netherlands', 'Belgium', 'Croatia', 'Morocco', 'Colombia',
    'Uruguay', 'Denmark', 'Mexico', 'USA', 'Senegal', 'Japan', 'Switzerland'
]

wc_stats = team_stats[team_stats['team'].isin(wc_teams)].copy()

fig, ax = plt.subplots(figsize=(12, 8))

scatter = ax.scatter(
    wc_stats['goals_against_avg'],
    wc_stats['goals_for_avg'],
    s=wc_stats['win_rate'] * 300,
    c=wc_stats['goal_diff_avg'],
    cmap='RdYlGn',
    alpha=0.7,
    edgecolors='black',
    linewidth=0.5
)

for _, row in wc_stats.iterrows():
    ax.annotate(row['team'], (row['goals_against_avg'], row['goals_for_avg']),
                fontsize=8, ha='center', va='bottom', fontweight='bold')

ax.axhline(wc_stats['goals_for_avg'].mean(), color='gray', linestyle='--', alpha=0.5, label='Media ofensiva')
ax.axvline(wc_stats['goals_against_avg'].mean(), color='gray', linestyle=':', alpha=0.5, label='Media defensiva')

plt.colorbar(scatter, label='Diferencia de gol promedio')
ax.set_xlabel('Goles recibidos promedio (menor = mejor defensa)', fontsize=11)
ax.set_ylabel('Goles anotados promedio (mayor = mejor ataque)', fontsize=11)
ax.set_title('Fortaleza Ofensiva vs Defensiva - Equipos WC2026', fontsize=13, fontweight='bold')
ax.legend(fontsize=9)

ax.text(0.02, 0.98, 'Zona ideal\n(arriba-izquierda)', transform=ax.transAxes,
        fontsize=9, va='top', color='green', alpha=0.7)

plt.tight_layout()
plt.savefig('../data/wc2026/fortaleza_equipos.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Entrenamiento del Modelo Dixon-Coles

In [ ]:
# Entrenar el modelo
model = DixonColesModel()
model.fit(train_df)

# Parametros del modelo
print(f'\nParametros del modelo:')
print(f'  Ventaja de local   : {model.home_advantage:.4f}')
print(f'  Correccion rho     : {model.rho:.4f}')
print(f'  Equipos modelados  : {len(model.teams)}')

In [ ]:
# Ranking de fortaleza de los equipos del mundial
strengths = model.team_strengths()
wc_strengths = strengths[strengths['team'].isin(wc_teams + 
    ['Morocco', 'Ivory Coast', 'Ecuador', 'Peru', 'Australia', 'South Korea',
     'Turkey', 'Serbia', 'Austria', 'Chile', 'Colombia', 'Uruguay', 'Canada',
     'Mexico', 'USA'])].head(40)

# Grafico de barras horizontales
fig, ax = plt.subplots(figsize=(10, 12))

colors = plt.cm.RdYlGn(np.linspace(0.3, 0.9, len(wc_strengths)))
bars = ax.barh(range(len(wc_strengths)), wc_strengths['net_strength'].values,
               color=colors[::-1], edgecolor='black', linewidth=0.5)

ax.set_yticks(range(len(wc_strengths)))
ax.set_yticklabels(wc_strengths['team'].values, fontsize=9)
ax.invert_yaxis()
ax.set_xlabel('Fortaleza Neta (Ataque / Defensa)', fontsize=11)
ax.set_title('Ranking de Fortaleza - Modelo Dixon-Coles\nMundial 2026', 
             fontsize=13, fontweight='bold')
ax.axvline(1.0, color='red', linestyle='--', alpha=0.5, label='Media global')
ax.legend()

plt.tight_layout()
plt.savefig('../data/wc2026/ranking_fortaleza.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Prediccion de Partidos Individuales

In [ ]:
def plot_score_matrix(model, team1, team2, max_goals=6):
    """Visualiza la matriz de probabilidades de marcadores."""
    matrix = model.predict_score_matrix(team1, team2, neutral=True, max_goals=max_goals)
    pred = model.predict_outcome(team1, team2, neutral=True)
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # Heatmap de probabilidades
    sns.heatmap(
        matrix * 100, annot=True, fmt='.1f', cmap='YlOrRd',
        ax=axes[0], cbar_kws={'label': 'Probabilidad (%)'},
        xticklabels=range(max_goals+1), yticklabels=range(max_goals+1)
    )
    axes[0].set_xlabel(f'Goles {team2}', fontsize=11)
    axes[0].set_ylabel(f'Goles {team1}', fontsize=11)
    axes[0].set_title(f'Probabilidades de Marcadores\n{team1} vs {team2}', 
                      fontsize=12, fontweight='bold')
    
    # Probabilidades de resultado
    outcomes = ['Victoria\n' + team1, 'Empate', 'Victoria\n' + team2]
    probs = [pred['home_win_prob'] * 100, pred['draw_prob'] * 100, pred['away_win_prob'] * 100]
    colors_bar = ['#2ecc71', '#95a5a6', '#e74c3c']
    
    bars = axes[1].bar(outcomes, probs, color=colors_bar, edgecolor='black', linewidth=0.8)
    for bar, prob in zip(bars, probs):
        axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                    f'{prob:.1f}%', ha='center', va='bottom', fontweight='bold', fontsize=12)
    
    axes[1].set_ylabel('Probabilidad (%)', fontsize=11)
    axes[1].set_title(f'Probabilidades de Resultado\nMarcador predicho: {pred["most_likely_score"]}', 
                      fontsize=12, fontweight='bold')
    axes[1].set_ylim(0, max(probs) * 1.2)
    axes[1].text(0.5, 0.95, f'Goles esperados: {pred["expected_home_goals"]:.2f} - {pred["expected_away_goals"]:.2f}',
                transform=axes[1].transAxes, ha='center', fontsize=10, color='gray')
    
    plt.suptitle(f'{team1.upper()} vs {team2.upper()} | Mundial 2026',
                 fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()
    
    return pred

# Ejemplo: Argentina vs France (potencial final)
pred = plot_score_matrix(model, 'Argentina', 'France')
print(f"\nResumen: {pred['home_team']} {pred['most_likely_score']} {pred['away_team']}")
print(f"Ganador predicho: {pred['predicted_winner']}")

In [ ]:
# Analisis cabeza a cabeza
def show_h2h(team1, team2, results_df):
    h2h = compute_h2h_stats(results_df, team1, team2, last_n=15)
    print(f'\nHISTORIAL H2H: {team1} vs {team2} (ultimos {h2h["matches"]} partidos)')
    print(f'  Victorias {team1:20}: {h2h["team1_wins"]}')
    print(f'  Empates               : {h2h["draws"]}')
    print(f'  Victorias {team2:20}: {h2h["team2_wins"]}')
    print(f'  Goles promedio        : {h2h["team1_goals_avg"]:.2f} - {h2h["team2_goals_avg"]:.2f}')
    return h2h

show_h2h('Argentina', 'France', results_df)
show_h2h('Brazil', 'Germany', results_df)
show_h2h('Spain', 'England', results_df)

## 5. Prediccion Completa del Mundial 2026 (104 Partidos)

In [ ]:
# Ejecutar prediccion completa
predictor = TournamentPredictor(model)
predictions = predictor.run_full_prediction()

In [ ]:
# Ver tabla de posiciones
print('\nTABLA DE POSICIONES - TODOS LOS GRUPOS')
predictions['standings'][['group','pos','team','pts','w','d','l','gf','gc','gd']]

In [ ]:
# Ver partidos de la fase de grupos
print('\nPARTIDOS FASE DE GRUPOS - PRIMEROS 20')
predictions['group_matches'][['group','home_team','away_team',
    'predicted_score','home_win_prob','draw_prob','away_win_prob','result']].head(20)

In [ ]:
# Ver eliminatorias
print('\nELIMINATORIAS')
predictions['knockout_matches'][['phase','team1','team2',
    'predicted_winner','team1_win_prob','team2_win_prob','predicted_score']]

In [ ]:
# Visualizar probabilidades por ronda de equipos favoritos
knockout_df = predictions['knockout_matches']

if len(knockout_df) > 0:
    phases = knockout_df['phase'].unique()
    
    fig, ax = plt.subplots(figsize=(14, 6))
    
    for i, (_, row) in enumerate(knockout_df.iterrows()):
        t1_prob = row['team1_win_prob'] * 100
        t2_prob = row['team2_win_prob'] * 100
        
        label_left = f"{row['team1'][:12]} ({t1_prob:.0f}%)"
        label_right = f"{row['team2'][:12]} ({t2_prob:.0f}%)"
        
        color1 = '#2ecc71' if t1_prob > t2_prob else '#e74c3c'
        color2 = '#2ecc71' if t2_prob > t1_prob else '#e74c3c'
        
        ax.barh(i, t1_prob, color=color1, alpha=0.8, edgecolor='black', linewidth=0.3)
        ax.barh(i, -t2_prob, color=color2, alpha=0.8, edgecolor='black', linewidth=0.3)
        ax.text(t1_prob + 1, i, label_left, va='center', fontsize=7)
        ax.text(-t2_prob - 1, i, label_right, va='center', ha='right', fontsize=7)
    
    ax.set_yticks(range(len(knockout_df)))
    ax.set_yticklabels([f"{r['phase']}" for _, r in knockout_df.iterrows()], fontsize=8)
    ax.set_xlabel('Probabilidad de clasificacion (%)')
    ax.set_title('Probabilidades de Clasificacion - Eliminatorias WC2026', 
                 fontsize=13, fontweight='bold')
    ax.axvline(0, color='black', linewidth=1)
    ax.set_xlim(-105, 105)
    
    plt.tight_layout()
    plt.savefig('../data/wc2026/eliminatorias_probs.png', dpi=150, bbox_inches='tight')
    plt.show()

## 6. Exportar Predicciones

In [ ]:
# Exportar a Excel
output_file = predictor.export_predictions()
print(f'Predicciones guardadas en: {output_file}')

## 7. Analisis de Sensibilidad
¿Que tan sensible es el modelo a los datos de entrenamiento?

In [ ]:
# Comparar predicciones con distintas ventanas de tiempo
teams_to_compare = [('Argentina', 'France'), ('Brazil', 'Germany'), 
                    ('Spain', 'England'), ('Morocco', 'Portugal')]

results_sensitivity = []
for years in [4, 6, 8, 10]:
    train = prepare_training_data(results_df, years=years)
    m = DixonColesModel()
    m.fit(train)
    
    for t1, t2 in teams_to_compare:
        try:
            pred = m.predict_outcome(t1, t2, neutral=True)
            results_sensitivity.append({
                'years': years,
                'match': f'{t1} vs {t2}',
                'home_win': round(pred['home_win_prob'] * 100, 1),
                'draw': round(pred['draw_prob'] * 100, 1),
                'away_win': round(pred['away_win_prob'] * 100, 1),
                'score': pred['most_likely_score'],
            })
        except:
            pass

sens_df = pd.DataFrame(results_sensitivity)
print('Sensibilidad del modelo segun ventana temporal de entrenamiento:')
sens_df.pivot_table(index='match', columns='years', values='home_win')

In [ ]:
# Partido personalizado - cambia los equipos aqui
equipo1 = 'Brazil'    # <-- cambiar
equipo2 = 'Germany'   # <-- cambiar
es_eliminatoria = False  # True para eliminatoria con penaltis

if es_eliminatoria:
    pred = model.predict_knockout(equipo1, equipo2)
    print(f'{equipo1} gana: {pred["team1_win_prob"]*100:.1f}%')
    print(f'{equipo2} gana: {pred["team2_win_prob"]*100:.1f}%')
    print(f'Ganador predicho: {pred["most_likely_winner"]}')
else:
    pred = plot_score_matrix(model, equipo1, equipo2)
    show_h2h(equipo1, equipo2, results_df)